# 03 — Camada Gold: Modelagem analíticaModelo estrela com uma fato e três dimensões conformadas, mais duas tabelas agregadasorientadas às perguntas de negócio.```                    dim_tempo                        |   dim_gestor —— fato_informe —— dim_classe```As tabelas agregadas desta camada resolvem dois problemas identificados na análiseexploratória: a dupla contagem por estruturas feeder e as descontinuidades causadas porausência de reporte.

In [ ]:
from pyspark.sql import functions as Ffrom pyspark.sql import Windowfato = spark.table("workspace.silver.fato_informe")dim_classe = spark.table("workspace.silver.dim_classe")dim_tempo = spark.table("workspace.silver.dim_tempo")base = (fato.join(            dim_classe.select("cnpj_classe", "nome_classe", "classificacao_anbima",                              "credito_privado", "classe_feeder", "id_gestor",                              "gestor", "administrador"),            "cnpj_classe", "left")        .join(dim_tempo, "data", "left"))cp = base.filter("credito_privado")print(cp.count(), "linhas no recorte de credito privado")

## 3.1 Detecção de reporte incompletoA análise exploratória revelou uma queda de R$ 170 bi no patrimônio agregado em 13/01/2026,com recuperação em poucos dias e sem resgate correspondente. A causa era ausência de reportede seis classes de grande porte.O número total de classes não detecta esse caso: variou apenas de 4.011 para 4.007. Oindicador correto é quantas das maiores classes estão ausentes no dia — poucas classesgrandes movem o agregado muito mais que dezenas de pequenas.

In [ ]:
# As 100 maiores classes do recorte, por patrimônio médiograndes = (cp.groupBy("cnpj_classe")             .agg(F.avg("patrimonio_liquido").alias("pl_medio"))             .orderBy(F.desc("pl_medio")).limit(100))n_grandes = grandes.count()presenca = (cp.join(grandes.select("cnpj_classe"), "cnpj_classe")              .groupBy("data")              .agg(F.countDistinct("cnpj_classe").alias("grandes_presentes"))              .withColumn("grandes_ausentes", F.lit(n_grandes) - F.col("grandes_presentes"))              .withColumn("reporte_completo", F.col("grandes_ausentes") == 0))display(presenca.filter("not reporte_completo").orderBy(F.desc("grandes_ausentes")))

## 3.2 Fluxo diário agregadoTrês séries de patrimônio, para leituras diferentes:- `pl_total` — soma bruta, comparável a divulgações de mercado, mas com dupla contagem- `pl_total_ex_feeder` — exclui classes de investimento em cotas, aproximação do patrimônio  efetivamente alocado em ativos- `reporte_completo` — permite excluir dias incompletos das análises de série temporal

In [ ]:
diario = (cp.groupBy("data")    .agg(F.sum("patrimonio_liquido").alias("pl_total"),         F.sum(F.when(~F.col("classe_feeder"), F.col("patrimonio_liquido"))).alias("pl_total_ex_feeder"),         F.sum("captacao").alias("captacao_total"),         F.sum("resgate").alias("resgate_total"),         F.sum("captacao_liquida").alias("captacao_liquida_total"),         F.sum("cotistas").alias("cotistas_total"),         F.countDistinct("cnpj_classe").alias("n_classes"))    .join(presenca.select("data", "grandes_ausentes", "reporte_completo"), "data", "left")    .join(dim_tempo.select("data", "ano_mes", "ultimo_dia_util_mes",                           "ultimo_dia_util_trimestre", "dezembro"), "data", "left"))(diario.write.format("delta").mode("overwrite")      .option("overwriteSchema", True)      .saveAsTable("workspace.gold.fluxo_diario_credito_privado"))print(diario.count(), "dias |",      diario.filter("not reporte_completo").count(), "com reporte incompleto")

## 3.3 Resumo por classeO retorno é calculado com **janela ordenada por data**. Uma primeira versão usava `first` e`last` sem ordenação e produziu retornos médios negativos, implausíveis para renda fixa decrédito no período. O erro foi detectado por validação de plausibilidade contra oconhecimento do domínio, não por falha de execução.Ressalva registrada no catálogo: retorno medido por variação de cota subestima classes queamortizam ou distribuem rendimento.

In [ ]:
wf = (Window.partitionBy("cnpj_classe").orderBy("data")      .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))cotas = (cp    .withColumn("cota_inicial", F.first("valor_cota").over(wf))    .withColumn("cota_final", F.last("valor_cota").over(wf))    .withColumn("primeira_data", F.min("data").over(wf))    .withColumn("ultima_data", F.max("data").over(wf))    .select("cnpj_classe", "cota_inicial", "cota_final", "primeira_data", "ultima_data")    .dropDuplicates(["cnpj_classe"]))resumo = (cp.groupBy("cnpj_classe", "nome_classe", "classificacao_anbima",                     "id_gestor", "gestor", "classe_feeder")    .agg(F.avg("patrimonio_liquido").alias("pl_medio"),         F.max("patrimonio_liquido").alias("pl_maximo"),         F.sum("captacao").alias("captacao_periodo"),         F.sum("resgate").alias("resgate_periodo"),         F.sum("captacao_liquida").alias("captacao_liquida_periodo"),         F.avg("cotistas").alias("cotistas_medio"),         F.stddev("cotistas").alias("cotistas_desvio"),         F.count("*").alias("dias"))    .join(cotas, "cnpj_classe", "left")    .withColumn("retorno_periodo", F.col("cota_final") / F.col("cota_inicial") - 1)    .withColumn("dias_corridos", F.datediff("ultima_data", "primeira_data"))    .withColumn("cv_cotistas", F.col("cotistas_desvio") / F.col("cotistas_medio")))(resumo.write.format("delta").mode("overwrite")      .option("overwriteSchema", True)      .saveAsTable("workspace.gold.resumo_classe_credito_privado"))print(resumo.count(), "classes")

## 3.4 Catálogo de Dados no Unity CatalogDescrição de cada tabela e campo, com domínio de valores e linhagem, registrada no própriocatálogo. Os comentários carregam também as ressalvas metodológicas, de modo que quemconsultar a tabela encontre a limitação junto com o dado.

In [ ]:
%sqlCOMMENT ON TABLE workspace.silver.fato_informe IS'Informe diário tipado e deduplicado. Grão: classe/subclasse por dia útil. Origem: bronze.inf_diario_fi';ALTER TABLE workspace.silver.fato_informe ALTER COLUMN cnpj_classe COMMENT 'CNPJ da classe, apenas dígitos. Chave de join com dim_classe';ALTER TABLE workspace.silver.fato_informe ALTER COLUMN id_subclasse COMMENT 'Identificador da subclasse; vazio quando a classe não possui subclasses';ALTER TABLE workspace.silver.fato_informe ALTER COLUMN data COMMENT 'Data de competência. Chave de join com dim_tempo. Domínio: 2025-09-01 a 2026-07-31';ALTER TABLE workspace.silver.fato_informe ALTER COLUMN valor_carteira COMMENT 'Valor total da carteira em BRL';ALTER TABLE workspace.silver.fato_informe ALTER COLUMN patrimonio_liquido COMMENT 'Patrimônio líquido em BRL. Filtrado para > 0';ALTER TABLE workspace.silver.fato_informe ALTER COLUMN valor_cota COMMENT 'Valor da cota em BRL. Filtrado para > 0';ALTER TABLE workspace.silver.fato_informe ALTER COLUMN captacao COMMENT 'Captações do dia em BRL. Domínio: >= 0';ALTER TABLE workspace.silver.fato_informe ALTER COLUMN resgate COMMENT 'Resgates pagos no dia em BRL. Domínio: >= 0';ALTER TABLE workspace.silver.fato_informe ALTER COLUMN cotistas COMMENT 'Número de cotistas na data. Domínio: >= 0';ALTER TABLE workspace.silver.fato_informe ALTER COLUMN captacao_liquida COMMENT 'Derivado: captacao menos resgate. Pode ser negativo';

In [ ]:
%sqlCOMMENT ON TABLE workspace.silver.dim_classe IS'Dimensão de classe com atributos herdados da casca. Grão: uma linha por classe. Origem: bronze.cad_classe join bronze.cad_fundo por ID_Registro_Fundo';ALTER TABLE workspace.silver.dim_classe ALTER COLUMN cnpj_classe COMMENT 'CNPJ da classe, apenas dígitos. Chave primária';ALTER TABLE workspace.silver.dim_classe ALTER COLUMN classificacao_anbima COMMENT 'Classificação ANBIMA. Nulo para fundos estruturados e não adaptados à RCVM 175';ALTER TABLE workspace.silver.dim_classe ALTER COLUMN credito_privado COMMENT 'Derivado: TRUE se a classificação ANBIMA pertence ao recorte (Crédito Livre, Grau de Invest. duração livre, Grau de Invest. duração baixa)';ALTER TABLE workspace.silver.dim_classe ALTER COLUMN classe_feeder COMMENT 'Derivado por padrão na denominação (EM COTAS, FIC). Aproximação: identifica classes que aplicam em outras classes, fonte de dupla contagem no patrimônio agregado';ALTER TABLE workspace.silver.dim_classe ALTER COLUMN id_gestor COMMENT 'Chave substituta: MD5 do nome do gestor normalizado. Join com dim_gestor';ALTER TABLE workspace.silver.dim_classe ALTER COLUMN situacao COMMENT 'Situação cadastral na data de extração do cadastro';COMMENT ON TABLE workspace.silver.dim_gestor IS'Dimensão de gestor. Grão: um gestor. Origem: agregação de silver.dim_classe. Chave substituta por hash, pois a base da CVM não fornece identificador de gestor';COMMENT ON TABLE workspace.silver.dim_tempo IS'Dimensão de tempo construída a partir das datas presentes na fato. Grão: um dia útil com informe publicado. Atributos de fim de período usados na análise de sazonalidade de resgates';

In [ ]:
%sqlCOMMENT ON TABLE workspace.gold.fluxo_diario_credito_privado IS'Série diária agregada da indústria de crédito privado. Grão: um dia. Origem: silver.fato_informe join dim_classe e dim_tempo, filtrado por credito_privado';ALTER TABLE workspace.gold.fluxo_diario_credito_privado ALTER COLUMN pl_total COMMENT 'Soma bruta do patrimônio líquido. Contém dupla contagem por estruturas feeder';ALTER TABLE workspace.gold.fluxo_diario_credito_privado ALTER COLUMN pl_total_ex_feeder COMMENT 'Patrimônio excluindo classes de investimento em cotas. Aproximação do valor efetivamente alocado em ativos';ALTER TABLE workspace.gold.fluxo_diario_credito_privado ALTER COLUMN captacao_liquida_total COMMENT 'Soma de captacao menos resgate no dia. Pode ser negativo';ALTER TABLE workspace.gold.fluxo_diario_credito_privado ALTER COLUMN n_classes COMMENT 'Classes distintas com informe no dia';ALTER TABLE workspace.gold.fluxo_diario_credito_privado ALTER COLUMN grandes_ausentes COMMENT 'Quantas das 100 maiores classes não reportaram no dia. Detecta descontinuidade da série';ALTER TABLE workspace.gold.fluxo_diario_credito_privado ALTER COLUMN reporte_completo COMMENT 'Derivado: FALSE quando alguma das 100 maiores classes está ausente. Dias FALSE devem ser excluídos de análises de série temporal';COMMENT ON TABLE workspace.gold.resumo_classe_credito_privado IS'Resumo por classe no período. Grão: uma classe. Origem: silver.fato_informe join dim_classe';ALTER TABLE workspace.gold.resumo_classe_credito_privado ALTER COLUMN retorno_periodo COMMENT 'Derivado: cota_final sobre cota_inicial menos 1, com janela ordenada por data. Subestima classes que amortizam ou distribuem rendimento';ALTER TABLE workspace.gold.resumo_classe_credito_privado ALTER COLUMN cv_cotistas COMMENT 'Derivado: desvio padrão sobre média do número de cotistas. Proxy de estabilidade da base';ALTER TABLE workspace.gold.resumo_classe_credito_privado ALTER COLUMN dias COMMENT 'Dias com informe no período. Usado para excluir classes com histórico curto das comparações';